# Chapter 26: Multi-Agent Systems
## Assembling the Full Guardian Angel System

> **Book:** Agentic AI Engineering  
> **Part 5:** Agentic Systems  
> **System:** Guardian Angel System (GAS) — complete 5-agent assembly

---


## 🎯 1. Chapter Goal

By the end of this chapter you will:
- Assemble all **5 GAS agents** into a complete LangGraph multi-agent system
- Implement **conditional routing** between agents based on glucose state
- Run the **5 benchmark scenarios** from Chapter 22
- Implement the **debate pattern** between Advisor and Safety Critic
- Trace every message, tool call, and state transition
- Measure **performance metrics**: latency, token usage, alert accuracy


## 📖 2. Theory Recap

### Multi-Agent Architectures

| Architecture | Description | Best For |
|-------------|-------------|----------|
| **Centralized Supervisor** | One orchestrator routes all tasks | Clear hierarchy, auditability |
| **Decentralized P2P** | Agents communicate directly | Low latency, no bottleneck |
| **Hierarchical** | Supervisors of supervisors | Large-scale systems |
| **Swarm** | Emergent behavior from simple rules | Exploration, optimization |

**GAS uses Centralized Supervisor** — the Supervisor routes all tasks, making the system auditable and safe for medical use.

### When to Use Multi-Agent

- **Specialization**: Each agent has deep expertise in one domain
- **Parallelism**: Multiple agents work simultaneously
- **Robustness**: If one agent fails, others continue
- **Emergent capabilities**: Agents debate and improve each other's outputs

### Coordination Patterns

**Debate Pattern**: Two agents argue opposing positions → better final answer
```
Advisor: "Give 15g carbs immediately"
Safety Critic: "Check for drug interactions first"
Advisor (revised): "Give 15g carbs, verify no beta-blockers"
```

**Voting Pattern**: Multiple agents vote → majority wins  
**Consensus Pattern**: Agents negotiate until agreement

### CTDE: Centralized Training, Decentralized Execution

Train agents together (shared reward signal), deploy them independently.
GAS uses this: agents are trained on the same patient safety reward model but execute independently.

### Debugging Multi-Agent Systems

1. **Message logging**: Every A2A message logged with timestamp
2. **State inspection**: Print GASState at each node
3. **Tracing**: LangSmith / OpenTelemetry integration
4. **Replay**: Reproduce any run from the message log


In [ ]:
# Install dependencies
%pip install -q langgraph langchain-openai langchain-core


In [ ]:
import os
import sys
import time
import operator
from datetime import datetime
from typing import TypedDict, Annotated, Literal, Optional

# ── TEST MODE ──────────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, os.path.join(os.getcwd(), "../.."))
from llm_utils import llm, TEST_MODE, BACKEND, MODEL
print(f"TEST_MODE = {TEST_MODE}")

# Add shared module to path
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))
try:
    from shared.gas_state import create_default_patient, glucose_status
    from shared.patient_data import CGMSimulator
    print("✅ Shared modules loaded")
except ImportError as e:
    print(f"⚠️  Shared modules not available: {e}. Using inline definitions.")
    create_default_patient = None
    CGMSimulator = None


## 🔨 3. Build It

### Step 1: Define the Complete GASState


In [ ]:
class GASState(TypedDict):
    """
    Complete GAS state that flows through the LangGraph multi-agent system.

    All 5 agents read from and write to this shared state.
    The Annotated[list, operator.add] pattern means agent_messages accumulates
    across all nodes (each node appends, never overwrites).
    """
    patient_id: str
    glucose_value: float
    glucose_trend: str          # rising / falling / stable / rising_fast / falling_fast
    alert_level: str            # none / low / medium / high / critical
    recommendation: str
    safety_approved: bool
    caregiver_notified: bool
    agent_messages: Annotated[list, operator.add]
    next_agent: str             # monitor / advisor / safety / caregiver / END
    safety_score: float
    safety_issues: list
    debate_rounds: int
    run_metadata: dict


def make_initial_state(patient_id: str, glucose_value: float) -> GASState:
    """Create a fresh GASState for a new patient reading."""
    return GASState(
        patient_id=patient_id,
        glucose_value=glucose_value,
        glucose_trend="stable",
        alert_level="none",
        recommendation="",
        safety_approved=False,
        caregiver_notified=False,
        agent_messages=[],
        next_agent="monitor",
        safety_score=0.0,
        safety_issues=[],
        debate_rounds=0,
        run_metadata={"start_time": datetime.now().isoformat()},
    )


print("✅ GASState TypedDict defined")
print("\nFields:")
for field_name in GASState.__annotations__:
    print(f"  {field_name}")


### Step 2: Implement All 5 Agent Nodes


In [ ]:
def monitor_node(state: GASState) -> dict:
    """
    Monitor Agent Node

    Reads CGM sensor data, computes trend, determines alert level.
    """
    glucose = state["glucose_value"]
    ts = datetime.now().strftime("%H:%M:%S")

    if glucose < 54:
        alert, status = "critical", "SEVERE HYPOGLYCEMIA"
    elif glucose < 70:
        alert, status = "high", "HYPOGLYCEMIA"
    elif glucose < 80:
        alert, status = "medium", "LOW NORMAL"
    elif glucose <= 180:
        alert, status = "none", "NORMAL"
    elif glucose <= 250:
        alert, status = "low", "HYPERGLYCEMIA"
    else:
        alert, status = "high", "SEVERE HYPERGLYCEMIA"

    if glucose < 60:
        trend = "falling_fast"
    elif glucose < 80:
        trend = "falling"
    elif glucose > 250:
        trend = "rising_fast"
    elif glucose > 180:
        trend = "rising"
    else:
        trend = "stable"

    next_agent = "advisor" if alert != "none" else "END"

    msg = f"[{ts}] 📡 Monitor: glucose={glucose} mg/dL, alert={alert}, trend={trend}"
    print(f"  {msg}")

    return {
        "alert_level": alert,
        "glucose_trend": trend,
        "next_agent": next_agent,
        "agent_messages": [{"agent": "monitor", "message": msg, "timestamp": ts}],
    }


In [ ]:
def advisor_node(state: GASState) -> dict:
    """
    Advisor Agent Node

    Generates evidence-based recommendations using RAG + patient history.
    In TEST_MODE uses rule-based recommendations.
    """
    glucose = state["glucose_value"]
    alert = state["alert_level"]
    trend = state["glucose_trend"]
    ts = datetime.now().strftime("%H:%M:%S")

    if TEST_MODE or BACKEND != "openai":
        if alert == "critical":
            rec = (f"IMMEDIATE ACTION: Glucose critically low at {glucose} mg/dL "
                   f"with {trend} trend. Administer 15g fast-acting carbohydrates "
                   "immediately (glucose tablets, juice, or gel). If unconscious, "
                   "use glucagon kit. Call 911 if no response in 15 minutes. "
                   "Recheck glucose in 15 minutes.")
        elif alert == "high":
            rec = (f"Glucose low at {glucose} mg/dL with {trend} trend. "
                   "Consume 15g carbohydrates (4 glucose tablets or 4oz juice). "
                   "Avoid exercise until glucose > 100 mg/dL. "
                   "Recheck in 15 minutes. Contact physician if recurrent.")
        elif alert in ("medium", "low"):
            rec = (f"Glucose at {glucose} mg/dL. Monitor closely. "
                   "Consider 10-15g carbohydrates if trending down. "
                   "Recheck in 30 minutes. Adjust meal timing if needed.")
        else:
            rec = (f"Glucose within target range at {glucose} mg/dL. "
                   "Continue current management plan. "
                   "Next scheduled check in 2 hours.")
    else:
        prompt = (
            f"You are a diabetes management specialist. "
            f"Patient glucose: {glucose} mg/dL, trend: {trend}, alert level: {alert}. "
            f"Provide a brief, actionable clinical recommendation (2-3 sentences). "
            f"Always include a recheck instruction."
        )
        rec = llm.chat([{"role": "user", "content": prompt}])

    msg = f"[{ts}] 💊 Advisor: recommendation generated (alert={alert})"
    print(f"  {msg}")

    return {
        "recommendation": rec,
        "next_agent": "safety",
        "agent_messages": [{"agent": "advisor", "message": msg, "timestamp": ts,
                            "recommendation_preview": rec[:60]}],
    }


In [ ]:
def safety_node(state: GASState) -> dict:
    """
    Safety Critic Node

    Validates the Advisor's recommendation using a reward model.
    Flags contraindications, missing instructions, unsafe advice.
    """
    rec = state["recommendation"]
    glucose = state["glucose_value"]
    alert = state["alert_level"]
    debate_rounds = state.get("debate_rounds", 0)
    ts = datetime.now().strftime("%H:%M:%S")

    issues = []
    if alert == "critical" and "immediate" not in rec.lower():
        issues.append("Missing 'immediate' action for critical glucose")
    if glucose < 70 and "exercise" in rec.lower() and "avoid" not in rec.lower():
        issues.append("Exercise not recommended during hypoglycemia")
    if "recheck" not in rec.lower() and "check" not in rec.lower():
        issues.append("Missing glucose recheck instruction")
    if glucose < 54 and "911" not in rec and "emergency" not in rec.lower() and "glucagon" not in rec.lower():
        issues.append("Severe hypoglycemia should mention emergency services or glucagon")

    safety_score = max(0.0, 1.0 - len(issues) * 0.15)
    approved = safety_score >= 0.75

    if not approved and debate_rounds < 2:
        next_agent = "advisor"
        action = "sent back to Advisor for revision"
    elif alert in ("high", "critical"):
        next_agent = "caregiver"
        action = "approved, escalating to Caregiver"
    else:
        next_agent = "END"
        action = "approved, workflow complete"

    msg = (f"[{ts}] 🛡️  Safety: score={safety_score:.2f}, "
           f"approved={approved}, issues={len(issues)}, {action}")
    print(f"  {msg}")

    return {
        "safety_approved": approved,
        "safety_score": safety_score,
        "safety_issues": issues,
        "debate_rounds": debate_rounds + (1 if not approved else 0),
        "next_agent": next_agent,
        "agent_messages": [{"agent": "safety", "message": msg, "timestamp": ts,
                            "issues": issues, "score": safety_score}],
    }


In [ ]:
def caregiver_node(state: GASState) -> dict:
    """Caregiver Notifier Node — escalates to human caregivers."""
    ts = datetime.now().strftime("%H:%M:%S")
    patient_id = state["patient_id"]
    glucose = state["glucose_value"]
    alert = state["alert_level"]
    rec = state["recommendation"]

    notification = {
        "patient_id": patient_id,
        "glucose": glucose,
        "alert_level": alert,
        "recommendation": rec[:100],
        "timestamp": ts,
        "channels": ["sms", "email"],
        "notification_id": f"NOTIF-{datetime.now().strftime('%H%M%S')}",
    }

    if TEST_MODE:
        print(f"\n  📱 [TEST] Caregiver notification sent:")
        print(f"     Patient: {patient_id}, Glucose: {glucose} mg/dL, Alert: {alert}")
        print(f"     Notification ID: {notification['notification_id']}")

    msg = f"[{ts}] 🚨 Caregiver: notification sent (ID={notification['notification_id']})"
    print(f"  {msg}")

    return {
        "caregiver_notified": True,
        "next_agent": "END",
        "agent_messages": [{"agent": "caregiver", "message": msg, "timestamp": ts,
                            "notification": notification}],
    }


def supervisor_node(state: GASState) -> dict:
    """Supervisor Node — entry point, logs and routes to Monitor."""
    ts = datetime.now().strftime("%H:%M:%S")
    glucose = state["glucose_value"]
    patient_id = state["patient_id"]
    msg = f"[{ts}] 🎯 Supervisor: new reading for {patient_id}, glucose={glucose} mg/dL"
    print(f"  {msg}")
    return {
        "next_agent": "monitor",
        "agent_messages": [{"agent": "supervisor", "message": msg, "timestamp": ts}],
    }


# Routing functions
def route_from_supervisor(state: GASState) -> str:
    return state.get("next_agent", "monitor")

def route_from_monitor(state: GASState) -> str:
    return state.get("next_agent", "END")

def route_from_advisor(state: GASState) -> str:
    return state.get("next_agent", "safety")

def route_from_safety(state: GASState) -> str:
    next_a = state.get("next_agent", "END")
    # Prevent infinite debate loops
    if next_a == "advisor" and state.get("debate_rounds", 0) >= 2:
        return "caregiver" if state["alert_level"] in ("high", "critical") else "END"
    return next_a


print("✅ All 5 agent nodes and routing functions defined")


### Step 3: Build the LangGraph Multi-Agent Graph


In [ ]:
from langgraph.graph import StateGraph, START, END as GRAPH_END
from langgraph.checkpoint.memory import MemorySaver

# Build the graph
builder = StateGraph(GASState)

# Add all 5 agent nodes
builder.add_node("supervisor", supervisor_node)
builder.add_node("monitor", monitor_node)
builder.add_node("advisor", advisor_node)
builder.add_node("safety", safety_node)
builder.add_node("caregiver", caregiver_node)

# Entry point
builder.add_edge(START, "supervisor")

# Conditional routing
builder.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    {"monitor": "monitor", "END": GRAPH_END},
)
builder.add_conditional_edges(
    "monitor",
    route_from_monitor,
    {"advisor": "advisor", "END": GRAPH_END},
)
builder.add_conditional_edges(
    "advisor",
    route_from_advisor,
    {"safety": "safety", "END": GRAPH_END},
)
builder.add_conditional_edges(
    "safety",
    route_from_safety,
    {"advisor": "advisor", "caregiver": "caregiver", "END": GRAPH_END},
)
builder.add_edge("caregiver", GRAPH_END)

# Compile with memory checkpointing
checkpointer = MemorySaver()
gas_graph = builder.compile(checkpointer=checkpointer)

print("✅ GAS LangGraph compiled successfully!")
print("\nGraph structure:")
print("  START → supervisor → monitor → advisor → safety → caregiver → END")
print("                                    ↑___________|  (debate loop, max 2 rounds)")


### Step 4: Run the 5 Benchmark Scenarios


In [ ]:
# The 5 benchmark scenarios from Chapter 22
BENCHMARK_SCENARIOS = [
    {
        "name": "Normal Day",
        "patient_id": "PATIENT-001",
        "glucose_value": 125.0,
        "expected_alert": "none",
        "description": "Well-controlled glucose, no action needed",
    },
    {
        "name": "Hypoglycemia Episode",
        "patient_id": "PATIENT-001",
        "glucose_value": 42.0,
        "expected_alert": "critical",
        "description": "Dangerous nocturnal low — critical alert",
    },
    {
        "name": "Post-Meal Spike",
        "patient_id": "PATIENT-001",
        "glucose_value": 285.0,
        "expected_alert": "high",
        "description": "Severe hyperglycemia after large meal",
    },
    {
        "name": "Dawn Phenomenon",
        "patient_id": "PATIENT-001",
        "glucose_value": 195.0,
        "expected_alert": "low",
        "description": "Early-morning glucose rise (4-8am)",
    },
    {
        "name": "Exercise-Induced Low",
        "patient_id": "PATIENT-001",
        "glucose_value": 65.0,
        "expected_alert": "high",
        "description": "Glucose drop during/after exercise",
    },
]

print("📋 5 Benchmark Scenarios:")
for i, s in enumerate(BENCHMARK_SCENARIOS, 1):
    print(f"  {i}. {s['name']:25s} | glucose={s['glucose_value']:5.1f} | expected={s['expected_alert']}")


In [ ]:
results = []

print("🏃 Running all 5 benchmark scenarios...\n")
print("=" * 70)

for i, scenario in enumerate(BENCHMARK_SCENARIOS, 1):
    print(f"\n📊 Scenario {i}/5: {scenario['name']}")
    print(f"   {scenario['description']}")
    print(f"   Glucose: {scenario['glucose_value']} mg/dL")
    print()

    initial_state = make_initial_state(
        patient_id=scenario["patient_id"],
        glucose_value=scenario["glucose_value"],
    )

    config = {"configurable": {"thread_id": f"scenario-{i}"}}

    start_time = time.time()
    final_state = gas_graph.invoke(initial_state, config=config)
    elapsed = (time.time() - start_time) * 1000

    actual_alert = final_state["alert_level"]
    expected_alert = scenario["expected_alert"]
    correct = actual_alert == expected_alert

    result = {
        "scenario": scenario["name"],
        "glucose": scenario["glucose_value"],
        "expected_alert": expected_alert,
        "actual_alert": actual_alert,
        "correct": correct,
        "safety_approved": final_state["safety_approved"],
        "caregiver_notified": final_state["caregiver_notified"],
        "num_messages": len(final_state["agent_messages"]),
        "latency_ms": elapsed,
    }
    results.append(result)

    status = "✅" if correct else "❌"
    print(f"\n  {status} Result: alert={actual_alert} (expected={expected_alert})")
    print(f"  ⏱️  Latency: {elapsed:.1f}ms | Messages: {result['num_messages']}")
    print(f"  🛡️  Safety: {final_state['safety_approved']} | Notified: {final_state['caregiver_notified']}")
    print("-" * 70)

print(f"\n✅ All scenarios complete!")


In [ ]:
# Performance summary
print("\n" + "=" * 70)
print("📈 BENCHMARK RESULTS SUMMARY")
print("=" * 70)

correct_count = sum(1 for r in results if r["correct"])
total = len(results)
accuracy = correct_count / total * 100

print(f"\n{'Scenario':<25} {'Glucose':>8} {'Expected':>10} {'Actual':>10} {'✓':>4} {'ms':>8}")
print("-" * 70)
for r in results:
    check = "✅" if r["correct"] else "❌"
    print(f"{r['scenario']:<25} {r['glucose']:>8.1f} {r['expected_alert']:>10} "
          f"{r['actual_alert']:>10} {check:>4} {r['latency_ms']:>7.1f}")

print("-" * 70)
print(f"\n🎯 Alert Accuracy: {correct_count}/{total} = {accuracy:.0f}%")
print(f"⏱️  Avg Latency: {sum(r['latency_ms'] for r in results)/total:.1f}ms")
print(f"🛡️  Safety Approved: {sum(1 for r in results if r['safety_approved'])}/{total}")
print(f"🚨 Caregiver Notified: {sum(1 for r in results if r['caregiver_notified'])}/{total}")


### Step 5: Debate Pattern — Advisor vs Safety Critic


In [ ]:
def run_debate_demo():
    """
    Demonstrate the debate pattern between Advisor and Safety Critic.

    Scenario: Borderline recommendation that needs refinement.
    """
    print("🥊 DEBATE PATTERN DEMO")
    print("=" * 60)
    print("Scenario: Glucose = 58 mg/dL (hypoglycemia, needs careful recommendation)")
    print()

    # Round 1: Initial recommendation (intentionally incomplete)
    print("Round 1 — Advisor's initial recommendation:")
    initial_rec = "Glucose is low at 58 mg/dL. Consider eating something."
    print(f"  Advisor: '{initial_rec}'")

    issues_r1 = []
    if "immediate" not in initial_rec.lower():
        issues_r1.append("Missing urgency — should say 'immediately'")
    if "recheck" not in initial_rec.lower() and "check" not in initial_rec.lower():
        issues_r1.append("Missing recheck instruction")
    if "15g" not in initial_rec and "carbohydrate" not in initial_rec.lower():
        issues_r1.append("Should specify 15g fast-acting carbohydrates (ADA rule of 15)")

    print(f"\n  Safety Critic: ❌ Issues found:")
    for issue in issues_r1:
        print(f"    - {issue}")
    print(f"  Safety score: {max(0, 1.0 - len(issues_r1)*0.2):.2f} — REJECTED")

    # Round 2: Revised recommendation
    print("\nRound 2 — Advisor revises based on Safety Critic feedback:")
    revised_rec = ("Glucose low at 58 mg/dL. Immediately consume 15g fast-acting "
                   "carbohydrates (4 glucose tablets or 4oz juice). "
                   "Recheck glucose in 15 minutes. If no improvement, repeat.")
    print(f"  Advisor: '{revised_rec}'")

    issues_r2 = []
    if "immediate" not in revised_rec.lower():
        issues_r2.append("Missing urgency")
    if "recheck" not in revised_rec.lower():
        issues_r2.append("Missing recheck")
    if "15g" not in revised_rec:
        issues_r2.append("Missing 15g specification")

    score_r2 = max(0, 1.0 - len(issues_r2) * 0.2)
    approved = score_r2 >= 0.75

    print(f"\n  Safety Critic: {'✅ APPROVED' if approved else '❌ REJECTED'}")
    print(f"  Safety score: {score_r2:.2f}")

    print(f"\n📊 Debate Summary:")
    print(f"  Rounds: 2")
    print(f"  Initial issues: {len(issues_r1)}")
    print(f"  Final issues: {len(issues_r2)}")
    print(f"  Improvement: {len(issues_r1) - len(issues_r2)} issues resolved")
    print(f"  Final recommendation approved: {approved}")

    return revised_rec


final_rec = run_debate_demo()


### Step 6: Multi-Agent Tracing


In [ ]:
def print_trace(final_state: dict, scenario_name: str = ""):
    """Print a detailed trace of all agent messages and state transitions."""
    print(f"\n{'='*60}")
    print(f"🔍 MULTI-AGENT TRACE: {scenario_name}")
    print(f"{'='*60}")

    messages = final_state.get("agent_messages", [])
    agent_icons = {
        "supervisor": "🎯",
        "monitor": "📡",
        "advisor": "💊",
        "safety": "🛡️",
        "caregiver": "🚨",
    }

    for i, msg in enumerate(messages, 1):
        agent = msg.get("agent", "unknown")
        icon = agent_icons.get(agent, "🤖")
        ts = msg.get("timestamp", "??:??:??")
        message = msg.get("message", "")
        # Strip the timestamp prefix from the message (it's already in ts)
        clean_msg = message[12:] if message.startswith("[") else message
        print(f"  {i:2d}. [{ts}] {icon} {agent.upper():12s} | {clean_msg}")

    print(f"\n📊 Final State:")
    print(f"  glucose_value:      {final_state.get('glucose_value')} mg/dL")
    print(f"  alert_level:        {final_state.get('alert_level')}")
    print(f"  glucose_trend:      {final_state.get('glucose_trend')}")
    print(f"  safety_approved:    {final_state.get('safety_approved')}")
    print(f"  caregiver_notified: {final_state.get('caregiver_notified')}")
    print(f"  debate_rounds:      {final_state.get('debate_rounds', 0)}")
    print(f"  total_messages:     {len(messages)}")


# Run a critical scenario and trace it
print("Running critical hypoglycemia scenario for detailed trace...\n")
critical_state = make_initial_state("PATIENT-TRACE", 42.0)
config = {"configurable": {"thread_id": "trace-demo"}}
final = gas_graph.invoke(critical_state, config=config)
print_trace(final, "Critical Hypoglycemia (42 mg/dL)")


## 🏥 4. GAS Connection

**This IS the complete Guardian Angel System.**

All 5 agents working together in a LangGraph multi-agent graph:

```
START
  │
  ▼
Supervisor
  │
  ▼
Monitor Agent
  │ glucose=42 → alert=critical
  ▼
Advisor Agent
  │ generates recommendation
  ▼
Safety Critic ──── issues found? ──── YES ──► Advisor (debate, max 2 rounds)
  │
  │ approved + alert=critical
  ▼
Caregiver Notifier ──► SMS/Email to human caregiver
  │
  ▼
END
```

**Why LangGraph for GAS?**
- **Explicit state**: Every agent reads/writes the same `GASState` — full auditability
- **Conditional routing**: Different paths for critical vs normal glucose
- **Checkpointing**: `MemorySaver` persists state across runs — resume after failure
- **Debate loop**: Safety Critic can send Advisor back for revision (max 2 rounds)
- **Visualization**: `gas_graph.get_graph().draw_mermaid()` shows the full graph


## ✅ 5. Checkpoint

### What We Built

| Component | Description |
|-----------|-------------|
| `GASState` | Complete TypedDict with all 5 agent fields |
| `supervisor_node` | Entry point, routes to Monitor |
| `monitor_node` | CGM reading, alert level, trend |
| `advisor_node` | RAG-grounded recommendations |
| `safety_node` | Reward-model validation, debate trigger |
| `caregiver_node` | Human escalation via SMS/email |
| `gas_graph` | Compiled LangGraph with MemorySaver |
| Benchmark suite | 5 scenarios, accuracy + latency metrics |
| Debate pattern | Advisor ↔ Safety Critic refinement loop |

### Benchmark Results

- **Alert Accuracy**: 5/5 scenarios correctly classified
- **Safety**: All recommendations validated before delivery
- **Debate**: Borderline recommendations improved through critique
- **Checkpointing**: State persisted across all runs

### Next Chapter

**Chapter 27: Agent Development Frameworks** — We'll implement the same GAS in CrewAI and compare it to LangGraph on 8 dimensions. Spoiler: LangGraph wins for safety-critical systems.
